# Decision-PGA: Fisher-Rao Probability Clouds

This notebook explores a model-free first prototype of Decision-PGA. Categorical probability vectors are mapped to the positive unit sphere with the square-root embedding, summarized with an intrinsic mean, and analyzed with principal geodesic dispersion in the tangent space.

The purpose is not to call a real model yet. The purpose is to verify that synthetic decision-state clouds have different geometric signatures even when entropy alone is similar.

In [ ]:
from pathlib import Path
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
FIGURE_DIR = PROJECT_ROOT / "notebooks" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

from decision_pga import pga_probability_cloud, synthetic_probability_cloud

## Synthetic Scenarios

The first examples are deliberately controlled:

- **stable:** confident distributions around one class;
- **binary ambiguity:** probability mass moves mostly between two alternatives;
- **diffuse uncertainty:** mass spreads across many classes;
- **boundary:** a smooth perturbation sweeps across a two-class decision boundary;
- **regime shift:** the dominant class changes halfway through a sequence.

In [ ]:
def entropy(probs):
    probs = np.asarray(probs, dtype=float)
    return -np.sum(probs * np.log(probs), axis=1)


scenario_names = [
    "stable",
    "binary_ambiguity",
    "diffuse_uncertainty",
    "boundary",
    "regime_shift",
]
clouds = {
    name: synthetic_probability_cloud(name, n_samples=160, n_classes=5, seed=42)
    for name in scenario_names
}
results = {name: pga_probability_cloud(cloud, label=name) for name, cloud in clouds.items()}

rows = []
for name in scenario_names:
    result = results[name]
    rows.append(
        (
            name,
            float(np.mean(entropy(clouds[name]))),
            result.total_dispersion,
            result.pc1_fraction,
            result.anisotropy_ratio,
            result.mean_margin,
        )
    )

header = f"{'scenario':<22} {'entropy':>9} {'dispersion':>12} {'pc1_frac':>10} {'anisotropy':>12} {'margin':>9}"
print(header)
print("-" * len(header))
for name, ent, dispersion, pc1, anisotropy, margin in rows:
    print(f"{name:<22} {ent:9.3f} {dispersion:12.4f} {pc1:10.3f} {anisotropy:12.3f} {margin:9.3f}")

## Cloud Shapes

The first plot shows the first two class probabilities. Binary ambiguity and boundary cases form coherent one-axis movement; diffuse uncertainty spreads more evenly.

In [ ]:
fig, axes = plt.subplots(1, len(scenario_names), figsize=(18, 3), sharex=True, sharey=True)
for ax, name in zip(axes, scenario_names):
    cloud = clouds[name]
    ax.scatter(cloud[:, 0], cloud[:, 1], s=14, alpha=0.65)
    mean = results[name].mean_probability
    ax.scatter([mean[0]], [mean[1]], s=80, marker="x", color="black")
    ax.set_title(name.replace("_", " "))
    ax.set_xlabel("p(class 0)")
axes[0].set_ylabel("p(class 1)")
fig.suptitle("Synthetic probability clouds")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "synthetic_probability_clouds.png", dpi=160)
plt.close(fig)

## Entropy Versus Geodesic Dispersion

Entropy can say that a cloud is uncertain, but Decision-PGA distinguishes whether uncertainty is concentrated along one coherent axis or spread across several directions.

In [ ]:
metric_names = ["entropy", "total_dispersion", "pc1_fraction", "mean_margin"]
metric_values = np.array([[row[1], row[2], row[3], row[5]] for row in rows], dtype=float)
scaled = metric_values / np.maximum(metric_values.max(axis=0, keepdims=True), 1e-12)

x = np.arange(len(scenario_names))
width = 0.20
fig, ax = plt.subplots(figsize=(11, 4))
for i, metric in enumerate(metric_names):
    ax.bar(x + (i - 1.5) * width, scaled[:, i], width, label=metric)
ax.set_xticks(x)
ax.set_xticklabels([name.replace("_", "\n") for name in scenario_names])
ax.set_ylabel("scaled metric value")
ax.set_title("Entropy compared with Decision-PGA diagnostics")
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "entropy_vs_decision_pga_metrics.png", dpi=160)
plt.close(fig)

## Sliding-Window Regime Shift

A later agent tool can run this kind of local analysis over tokens, prompt variants, sampled answers, or agent steps. Here, a synthetic regime shift creates a jump in the mean decision state and a local change in dispersion.

In [ ]:
regime = clouds["regime_shift"]
window = 28
centers = []
pc1 = []
dispersion = []
mean_class0 = []
mean_class2 = []
for start in range(0, len(regime) - window + 1):
    local = regime[start : start + window]
    result = pga_probability_cloud(local)
    centers.append(start + window / 2)
    pc1.append(result.pc1_fraction)
    dispersion.append(result.total_dispersion)
    mean_class0.append(result.mean_probability[0])
    mean_class2.append(result.mean_probability[2])

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(centers, mean_class0, label="mean p(class 0)")
axes[0].plot(centers, mean_class2, label="mean p(class 2)")
axes[0].axvline(len(regime) / 2, color="black", linestyle="--", alpha=0.5)
axes[0].set_ylabel("local mean probability")
axes[0].legend()
axes[1].plot(centers, dispersion, label="total dispersion")
axes[1].plot(centers, pc1, label="PC1 fraction")
axes[1].axvline(len(regime) / 2, color="black", linestyle="--", alpha=0.5)
axes[1].set_xlabel("sample index")
axes[1].set_ylabel("local metric")
axes[1].legend()
fig.suptitle("Sliding-window Decision-PGA over a synthetic regime shift")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "sliding_window_regime_shift.png", dpi=160)
plt.close(fig)

## Future Real-Model Adapter

A real model adapter should provide probability clouds from output logprobs, sampled answer probabilities, or calibrated candidate-action distributions. That adapter is intentionally out of scope for this first notebook; the core geometry should remain testable without API keys or hidden activations.